# Z-Score Tree Visualizations

This notebook generates hierarchical visualizations of Z-scores for various clinical terms (age, dose, gender, indications, etc.) across different models, helping interpret which terms are most strongly associated with true predictions.

In [11]:
import numpy as np
import pandas as pd
from scipy import stats
from tqdm import tqdm
import os
import json
import shutup; shutup.please()
import networkx as nx
import matplotlib.pyplot as plt

In [12]:
def calculate_z_scores(df, terms_dict, score_col):
    """
    Calculate Z-scores for unique terms in specified columns of a DataFrame.
    
    Args:
    df (pd.DataFrame): The input DataFrame containing the data.
    terms_dict (dict): A dictionary where keys are column names and values are lists of unique terms to analyze.
    
    Returns:
    pd.DataFrame: A DataFrame containing columns, terms, and their corresponding Z-scores, sorted by Z-score.
    """
    results = []

    for column_name, terms in terms_dict.items():
        # Initialize a dictionary to store the Z-scores for this column
        z_scores = {}
        
        # Filter out rows where the specified column is NaN
        filtered_df = df[df[column_name].notna()]

        # Iterate through each term in the list
        for term in terms:
            # Create a mask for the rows containing the term
            mask_present = filtered_df[column_name].str.contains(term, na=False, regex=False)
            
            # Get probabilities for both parts
            prob_present = filtered_df.loc[mask_present, score_col]
            prob_absent = filtered_df.loc[~mask_present, score_col]
            
            # Check if we have enough data to perform the test
            if len(prob_present) >= 30 and len(prob_absent) >= 30:
                # Calculate means and standard deviations
                mean_present = prob_present.mean()
                mean_absent = prob_absent.mean()
                std_present = prob_present.std(ddof=1)  # Sample standard deviation
                std_absent = prob_absent.std(ddof=1)    # Sample standard deviation
                
                # Calculate the Z-score
                z_score = (mean_present - mean_absent) / np.sqrt((std_present**2 / len(prob_present)) + (std_absent**2 / len(prob_absent)))
                
                # Store the result
                z_scores[term] = z_score

        # Order Z-scores dictionary by values
        z_scores = dict(sorted(z_scores.items(), key=lambda item: item[1], reverse=True))

        # Append results for this column to the results list
        for term, z_score in z_scores.items():
            results.append({'Term': column_name + '_' + term, 'Z-score': z_score})

    # Convert the results list into a DataFrame
    z_scores_df = pd.DataFrame(results)

    # Sort the DataFrame by Z-score in descending order
    try:
        z_scores_df = z_scores_df.sort_values(by='Z-score', ascending=False)
        # drop rows where Z_score is less then 1.64 or nan
        z_scores_df = z_scores_df[(z_scores_df['Z-score'] > 1.64) | (z_scores_df['Z-score'].isna())]
    except:
        z_scores_df = None

    return z_scores_df

In [13]:
def create_and_plot_graph(dirpath, model, crossval_idx):
    xgb = pd.read_csv(os.path.join(dirpath,"cross_val_xgb", f"df_res{crossval_idx[0]}{crossval_idx[1]}.csv"), index_col=0)
    xgb.drop("label", axis=1, inplace=True)
    biobert_temp = pd.read_csv(os.path.join(dirpath,"cross_val_biobert_temp", f"df_res{crossval_idx[0]}{crossval_idx[1]}.csv"), index_col=0)
    biobert_temp.drop("label", axis=1, inplace=True)
    albert_temp = pd.read_csv(os.path.join(dirpath,"cross_val_albert_temp", f"df_res{crossval_idx[0]}{crossval_idx[1]}.csv"), index_col=0)
    albert_temp.drop("label", axis=1, inplace=True)
    llama_temp = pd.read_csv(os.path.join(dirpath, 'cross_val_llama_temp', f"df_res{crossval_idx[0]}{crossval_idx[1]}.csv"), index_col=0)
    llama_temp.drop("label", axis=1, inplace=True)
    df = pd.read_csv(os.path.join(dirpath,'df_together.csv'), index_col=0)
    df.drop("Temp_sentence", axis=1, inplace=True)
    df = df.loc[llama_temp.index]
    df = pd.concat([df, xgb, biobert_temp, albert_temp, llama_temp], axis=1)

    age_terms = list(set(list(df["age"].dropna())))
    dose_terms = list(set(list(df["dose"].dropna())))
    gender_terms = ["Male","Female"]

    # Function to get sorted unique terms from a specified column
    def get_sorted_unique_terms(column):
        terms = df[column].dropna().str.split(',').explode()  # Split by commas and explode
        terms = terms.str.strip()                     # Strip leading/trailing whitespace
        unique_terms = terms.unique().tolist()        # Get unique terms as a list
        return sorted(unique_terms)                   # Sort the terms alphabetically

    # Get sorted unique terms from each specified column
    sorted_unique_terms_psd = get_sorted_unique_terms('psd')
    sorted_unique_terms_ssd = get_sorted_unique_terms('ssd')
    sorted_unique_terms_ccd = get_sorted_unique_terms('ccd')
    sorted_unique_terms_idrug = get_sorted_unique_terms('idrug')
    sorted_unique_terms_indication = get_sorted_unique_terms('indication')
    if dirpath =='ailf':
        sorted_unique_terms_outcome = get_sorted_unique_terms('outcome')
    else:
        sorted_unique_terms_ade = get_sorted_unique_terms('ade')
    sorted_unique_age_terms = sorted(age_terms)
    sorted_unique_dose_terms = sorted(dose_terms)
    sorted_unique_gender_terms = sorted(gender_terms)

    if dirpath =='ailf':
        terms_dict = {
            'psd': sorted_unique_terms_psd,
            'ssd': sorted_unique_terms_ssd,
            'ccd': sorted_unique_terms_ccd,
            'idrug': sorted_unique_terms_idrug,
            'indication': sorted_unique_terms_indication,
            'outcome': sorted_unique_terms_outcome,
            'age': sorted_unique_age_terms,
            'dose': sorted_unique_dose_terms,
            'gender': sorted_unique_gender_terms
        }
    else:
        terms_dict = {
            'psd': sorted_unique_terms_psd,
            'ssd': sorted_unique_terms_ssd,
            'ccd': sorted_unique_terms_ccd,
            'idrug': sorted_unique_terms_idrug,
            'indication': sorted_unique_terms_indication,
            'ade': sorted_unique_terms_ade,
            'age': sorted_unique_age_terms,
            'dose': sorted_unique_dose_terms,
            'gender': sorted_unique_gender_terms
        }
    
    zscoresdf = calculate_z_scores(df, terms_dict, score_col=model)

    first_term = { 'Term': zscoresdf['Term'].iloc[0], 'Z-score': float(zscoresdf['Z-score'].iloc[0]) }

    colname, term_value = first_term['Term'].split('_', 1)
    reduced_df = df[(df[colname].str.contains(term_value, na=False))]

    zscoresdf = calculate_z_scores(reduced_df, terms_dict, score_col=model)

    second_terms = {
        0: {  'Term': zscoresdf['Term'].iloc[0], 'Z-score': float(zscoresdf['Z-score'].iloc[0]) },
        1: { 'Term': zscoresdf['Term'].iloc[1], 'Z-score': float(zscoresdf['Z-score'].iloc[1]) },
    }

    colname0, term_value0 = second_terms[0]['Term'].split('_', 1)
    reduced_df0 = reduced_df[(reduced_df[colname0].str.contains(term_value0, na=False))]
    zscoresdf0 = calculate_z_scores(reduced_df0, terms_dict, score_col=model)
    if zscoresdf0 is not None:
        zscoresdf0 = zscoresdf0.rename(columns={'Z-score': 'Z-score 0'})

    colname1, term_value1 = second_terms[1]['Term'].split('_', 1)
    reduced_df1 = reduced_df[(reduced_df[colname1].str.contains(term_value1, na=False))]
    zscoresdf1 = calculate_z_scores(reduced_df1, terms_dict, score_col=model)

    if zscoresdf1 is not None:
        zscoresdf1 = zscoresdf1.rename(columns={'Z-score': 'Z-score 1'})
    if zscoresdf0 is not None:
        #remove rows from zscoresdf0 where Term is 2nd term
        zscoresdf0 = zscoresdf0[zscoresdf0['Term'] != second_terms[1]['Term']]
    if zscoresdf1 is not None:
        #remove rows from zscoresdf1 where Term is 2nd term
        zscoresdf1 = zscoresdf1[zscoresdf1['Term'] != second_terms[0]['Term']]

    dfs_to_concat = []
    if zscoresdf0 is not None:
        dfs_to_concat.append(zscoresdf0.set_index('Term'))
    if zscoresdf1 is not None:
        dfs_to_concat.append(zscoresdf1.set_index('Term'))
    if dfs_to_concat:
        combined_zscores = pd.concat(dfs_to_concat, axis=1)
    else:
        combined_zscores = None
    if combined_zscores is not None:
        zscore_cols = [col for col in combined_zscores.columns if col.startswith('Z-score')]
        combined_zscores['max Z-score'] = combined_zscores[zscore_cols].max(axis=1)
        combined_zscores = combined_zscores.sort_values(by='max Z-score', ascending=False)
        combined_zscores.drop('max Z-score', axis=1, inplace=True)
        combined_zscores.reset_index(inplace=True)
        # keep only the top 3 rows
        combined_zscores = combined_zscores.head(3)
        if 'Z-score 1' not in combined_zscores.columns:
            combined_zscores['Z-score 1'] = np.nan
        if 'Z-score 0' not in combined_zscores.columns:
            combined_zscores['Z-score 0'] = np.nan
    # Create NetworkX directed graph
    G = nx.DiGraph()

    # Add nodes
    # Bottom nodes: top 3 from combined_zscores
    if combined_zscores is not None:
        for term in combined_zscores['Term']:
            G.add_node(term)

    # Second terms nodes
    G.add_node(second_terms[0]['Term'])
    G.add_node(second_terms[1]['Term'])

    # First term node
    G.add_node(first_term['Term'])

    # Dataset node
    G.add_node(dirpath)
    if combined_zscores is not None:
        # Add edges from bottom to second terms
        for idx, row in combined_zscores.iterrows():
            term = row['Term']
            z0 = row['Z-score 0']
            z1 = row['Z-score 1']
            if not pd.isna(z0):
                G.add_edge(term, second_terms[0]['Term'], weight=z0)
            if not pd.isna(z1):
                G.add_edge(term, second_terms[1]['Term'], weight=z1)

    # Edges from second terms to first term
    G.add_edge(second_terms[0]['Term'], first_term['Term'], weight=second_terms[0]['Z-score'])
    G.add_edge(second_terms[1]['Term'], first_term['Term'], weight=second_terms[1]['Z-score'])

    # Edge from first term to dataset
    G.add_edge(first_term['Term'], dirpath, weight=first_term['Z-score'])

    # Plot the graph

    if combined_zscores is not None:
        # Define node levels for hierarchical layout
        levels = {
            0: list(combined_zscores['Term']),  # Bottom level: top 3 terms
            1: [second_terms[0]['Term'], second_terms[1]['Term']],  # Second level: second terms
            2: [first_term['Term']],  # Third level: first term
            3: [dirpath]  # Top level: dataset
        }
    if combined_zscores is None:
        levels = {  # Bottom level: top 3 terms
            0: [first_term['Term']],  # Bottom level: first term
            1: [second_terms[0]['Term'], second_terms[1]['Term']],  # Second level: second terms
            2: [dirpath],  # Third level: dataset
        }
    # Create positions for nodes in a pyramid-like layout
    pos = {}
    y_positions = {0: 0, 1: 1, 2: 2, 3: 3}  # Y-coordinates for levels
    x_spacing = 1.0  # Spacing between nodes on the same level

    for level, nodes in levels.items():
        num_nodes = len(nodes)
        x_start = -(num_nodes - 1) * x_spacing / 2
        for i, node in enumerate(nodes):
            pos[node] = (x_start + i * x_spacing, y_positions[level])

    # Draw the graph
    plt.figure(figsize=(10, 8))
    nx.draw(G, pos, with_labels=True, node_color='lightblue', node_size=3000, font_size=6, font_weight='bold', arrows=True, arrowstyle='->', arrowsize=20)

    # Add edge labels (Z-scores)
    edge_labels = {(u, v): f"{d['weight']:.2f}" for u, v, d in G.edges(data=True)}
    nx.draw_networkx_edge_labels(G, pos, edge_labels, font_size=8)

    plt.title('Hierarchical Pyramid Graph of Z-Scores')
    
    # Save the plot
    os.makedirs(os.path.join(dirpath, 'trees', model), exist_ok=True)
    plt.savefig(os.path.join(dirpath, 'trees', model, f'graph_{crossval_idx}.pdf'))
    plt.close()  # Close to free memory

In [ ]:
for dirpath in ['tram']:    
    for model in tqdm(['albert_temp','albert_temp_cal', 'xgb', 'xgb_cal', 'biobert_temp' ,'biobert_temp_cal', 'med_llama_temp', 'med_llama_temp_cal']):
        for i in range(5):
            for j in range(5):
                if i!=j:
                    try:
                        create_and_plot_graph(dirpath, model, f'{i}{j}')
                    except:
                        pass

  0%|          | 0/8 [00:00<?, ?it/s]